In [1]:
!pip install nltk

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.7/800.7 kB 5.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]1/2 [nltk]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [4]:
document = """
About Long Short-Term Memory (LSTM)

What is an LSTM?
LSTM (Long Short-Term Memory) is a special type of Recurrent Neural Network (RNN) designed to learn long-term dependencies in sequential data. It overcomes the vanishing gradient problem of traditional RNNs.

Why was LSTM introduced?
Traditional RNNs struggle to remember information over long sequences due to vanishing gradients. LSTMs introduce memory cells and gating mechanisms to preserve important information for longer durations.

Who invented LSTM?
LSTM was introduced by Sepp Hochreiter and Jürgen Schmidhuber in 1997.

What problems can LSTMs solve?
LSTMs are commonly used for:
- Text classification
- Sentiment analysis
- Machine translation
- Speech recognition
- Time-series forecasting
- Handwriting recognition
- Stock price prediction
- Music generation

What is the main difference between RNN and LSTM?
An RNN only maintains a hidden state, while an LSTM maintains both a hidden state and a cell state, allowing it to remember information over long sequences.

What are the main components of an LSTM cell?
An LSTM consists of:
- Forget Gate
- Input Gate
- Candidate Cell State
- Output Gate
- Hidden State
- Cell State

What is the Cell State?
The cell state is the long-term memory of the LSTM. It carries important information through the sequence with minimal modification.

What is the Hidden State?
The hidden state is the short-term memory and is used for producing outputs and passing information to the next time step.

Forget Gate

What is the purpose of the Forget Gate?
The Forget Gate decides what information should be removed from the previous cell state.

Which activation function does the Forget Gate use?
The Forget Gate uses the Sigmoid activation function.

What is the range of values produced by the Forget Gate?
The Forget Gate outputs values between 0 and 1.

What does a Forget Gate value of 0 mean?
Forget the information completely.

What does a Forget Gate value of 1 mean?
Retain all the information.

Input Gate

What is the purpose of the Input Gate?
The Input Gate determines which new information should be added to the cell state.

Which activation function does the Input Gate use?
The Input Gate uses the Sigmoid activation function.

Candidate Memory

What is Candidate Memory?
Candidate Memory contains new information that could potentially be stored in the cell state.

Which activation function generates Candidate Memory?
The tanh activation function.

What is the range of tanh?
The tanh activation function outputs values between -1 and 1.

Output Gate

What is the purpose of the Output Gate?
The Output Gate determines what part of the cell state should be exposed as the hidden state.

Which activation function does the Output Gate use?
The Sigmoid activation function.

Training an LSTM

What loss function is commonly used for LSTM classification?
CrossEntropyLoss.

What optimizer is commonly used?
Adam optimizer.

What learning rate is commonly used?
A learning rate of 0.001 is a common starting point.

What batch size is commonly used?
Typical batch sizes are 16, 32, or 64 depending on available memory.

What hidden sizes are commonly used?
Common hidden sizes include 64, 128, 256, and 512.

How many LSTM layers are commonly used?
One to three layers are common for most applications.

Should dropout be used?
Yes, dropout is recommended when using multiple LSTM layers to reduce overfitting.

What does batch_first=True mean in PyTorch?
The input tensor shape becomes:
(batch_size, sequence_length, input_size)

PyTorch LSTM

How do you create an LSTM in PyTorch?
Example:
nn.LSTM(
    input_size=100,
    hidden_size=128,
    num_layers=2,
    batch_first=True
)

What is input_size?
The number of input features at each time step.

What is hidden_size?
The number of features in the hidden state.

What is num_layers?
The number of stacked LSTM layers.

What is bidirectional=True?
The sequence is processed in both forward and backward directions.

What is dropout?
Dropout randomly deactivates neurons during training to reduce overfitting.

Input Shapes

What is the input shape when batch_first=True?
(batch_size, sequence_length, input_size)

What is the hidden state shape?
(num_layers * directions, batch_size, hidden_size)

What is the cell state shape?
(num_layers * directions, batch_size, hidden_size)

What does the output tensor contain?
The hidden outputs for every time step from the final LSTM layer.

Applications

Where are LSTMs used?
LSTMs are used in:
- Sentiment Analysis
- Language Modeling
- Text Generation
- Machine Translation
- Speech Recognition
- Time Series Prediction
- Financial Forecasting
- Weather Forecasting
- Video Analysis

Advantages

What are the advantages of LSTM?
- Learns long-term dependencies
- Handles sequential data efficiently
- Reduces vanishing gradient problems
- Suitable for time-series and NLP tasks

Limitations

What are the limitations of LSTM?
- Slower training compared to GRU
- More parameters than RNN
- Computationally expensive
- Difficult to parallelize
- Transformers outperform LSTMs on many NLP tasks

LSTM vs RNN

What is the biggest advantage of LSTM over RNN?
LSTMs can retain important information for much longer sequences using memory cells and gates.

LSTM vs GRU

What is the main difference between LSTM and GRU?
LSTM has three gates and a separate cell state, whereas GRU has two gates and combines the hidden state with memory.

Best Practices

What are some best practices while training an LSTM?
- Normalize numerical data.
- Use embedding layers for text inputs.
- Clip gradients to avoid exploding gradients.
- Use dropout when stacking multiple layers.
- Tune hidden size and learning rate.
- Use packed sequences for variable-length inputs.
- Monitor validation loss to avoid overfitting.

Common Interview Questions

Why does LSTM perform better than RNN?
Because its gating mechanism preserves important information and mitigates the vanishing gradient problem.

Can LSTM process variable-length sequences?
Yes, using pack_padded_sequence and pad_packed_sequence in PyTorch.

Can LSTM be used for time-series forecasting?
Yes, LSTMs are widely used for forecasting stock prices, weather, sales, and sensor data.

When should I use an LSTM?
Use an LSTM when your data is sequential and long-term dependencies are important, especially for time-series analysis and classical NLP tasks.
"""

In [3]:
# Tokernization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/foolmann/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/foolmann/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
#tokenize

tokens = word_tokenize(document.lower())

In [12]:
# build vocabie list of unique tokens

vocab = {'<unk>':0}

# Counter(tokens)
# Counter(tokens).keys()
for token in Counter(tokens).keys():
    # print(token)
    if token not in vocab:
        vocab[token]=len(vocab)
print(vocab)

{'<unk>': 0, 'about': 1, 'long': 2, 'short-term': 3, 'memory': 4, '(': 5, 'lstm': 6, ')': 7, 'what': 8, 'is': 9, 'an': 10, '?': 11, 'a': 12, 'special': 13, 'type': 14, 'of': 15, 'recurrent': 16, 'neural': 17, 'network': 18, 'rnn': 19, 'designed': 20, 'to': 21, 'learn': 22, 'long-term': 23, 'dependencies': 24, 'in': 25, 'sequential': 26, 'data': 27, '.': 28, 'it': 29, 'overcomes': 30, 'the': 31, 'vanishing': 32, 'gradient': 33, 'problem': 34, 'traditional': 35, 'rnns': 36, 'why': 37, 'was': 38, 'introduced': 39, 'struggle': 40, 'remember': 41, 'information': 42, 'over': 43, 'sequences': 44, 'due': 45, 'gradients': 46, 'lstms': 47, 'introduce': 48, 'cells': 49, 'and': 50, 'gating': 51, 'mechanisms': 52, 'preserve': 53, 'important': 54, 'for': 55, 'longer': 56, 'durations': 57, 'who': 58, 'invented': 59, 'by': 60, 'sepp': 61, 'hochreiter': 62, 'jürgen': 63, 'schmidhuber': 64, '1997.': 65, 'problems': 66, 'can': 67, 'solve': 68, 'are': 69, 'commonly': 70, 'used': 71, ':': 72, '-': 73, 'tex

In [13]:
len(vocab)

313

In [24]:
# sentences extraction from the data 

# document.split('\n')
# document.split('\n')[1]
input_sentences=[]
for sentence in document.split('\n'):
    if  sentence !=' ' and sentence != "":
        input_sentences.append(sentence)
print(input_sentences)


['About Long Short-Term Memory (LSTM)', 'What is an LSTM?', 'LSTM (Long Short-Term Memory) is a special type of Recurrent Neural Network (RNN) designed to learn long-term dependencies in sequential data. It overcomes the vanishing gradient problem of traditional RNNs.', 'Why was LSTM introduced?', 'Traditional RNNs struggle to remember information over long sequences due to vanishing gradients. LSTMs introduce memory cells and gating mechanisms to preserve important information for longer durations.', 'Who invented LSTM?', 'LSTM was introduced by Sepp Hochreiter and Jürgen Schmidhuber in 1997.', 'What problems can LSTMs solve?', 'LSTMs are commonly used for:', '- Text classification', '- Sentiment analysis', '- Machine translation', '- Speech recognition', '- Time-series forecasting', '- Handwriting recognition', '- Stock price prediction', '- Music generation', 'What is the main difference between RNN and LSTM?', 'An RNN only maintains a hidden state, while an LSTM maintains both a hi

In [26]:
def text_to_indices(sentence, vocab):
    numerical_sentence=[]

    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence

In [32]:

# for sentence in input_sentences:
#     print(text_to_indices(word_tokenize(sentence.lower()),vocab))

numerical_input_sentences = []
for sentence in input_sentences:
    # print(text_to_indices(word_tokenize(sentence.lower()),vocab))
    numerical_input_sentences.append(text_to_indices(word_tokenize(sentence.lower()),vocab))

print(numerical_input_sentences)



[[1, 2, 3, 4, 5, 6, 7], [8, 9, 10, 6, 11], [6, 5, 2, 3, 4, 7, 9, 12, 13, 14, 15, 16, 17, 18, 5, 19, 7, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 15, 35, 36, 28], [37, 38, 6, 39, 11], [35, 36, 40, 21, 41, 42, 43, 2, 44, 45, 21, 32, 46, 28, 47, 48, 4, 49, 50, 51, 52, 21, 53, 54, 42, 55, 56, 57, 28], [58, 59, 6, 11], [6, 38, 39, 60, 61, 62, 50, 63, 64, 25, 0, 28], [8, 66, 67, 47, 68, 11], [47, 69, 70, 71, 55, 72], [73, 74, 75], [73, 76, 77], [73, 78, 79], [73, 80, 81], [73, 82, 83], [73, 84, 81], [73, 85, 86, 87], [73, 88, 89], [8, 9, 31, 90, 91, 92, 19, 50, 6, 11], [10, 19, 93, 94, 12, 95, 96, 97, 98, 10, 6, 94, 99, 12, 95, 96, 50, 12, 100, 96, 97, 101, 29, 21, 41, 42, 43, 2, 44, 28], [8, 69, 31, 90, 102, 15, 10, 6, 100, 11], [10, 6, 103, 15, 72], [73, 104, 105], [73, 106, 105], [73, 107, 100, 96], [73, 108, 105], [73, 95, 96], [73, 100, 96], [8, 9, 31, 100, 96, 11], [31, 100, 96, 9, 31, 23, 4, 15, 31, 6, 28, 29, 109, 54, 42, 110, 31, 111, 112, 113, 114, 28], [8, 9, 31,

In [33]:
len(numerical_input_sentences)

154

In [ ]:
# # now forming the training sequence for each sentence ; 
# For ex 1,2,3,4,5,6,7 is the first sentence in indices 

# [1,2]
# [1,2,3]
# [1,2,3,4]
# # ie each new sequence has the input of previous . 
# Now we form the 

# input              output

# [1]                  [2]
# [1,2]                 [3]

# so we will be using this to use the end value as target 


training_sequence = []
for sentence in numerical_input_sentences:

    for i in range(1,len(sentence)):
        # print(sentence[:i+1])
        training_sequence.append(sentence[:i+1])


In [37]:
len(training_sequence)

974

In [47]:
# print(training_sequence)

In [39]:
# we need to make the len of the each sequence to be same so we are pre padding the 0 . 
len_list = []
for sequence in training_sequence:
    len_list.append(len(sequence))
max(len_list)

36

In [ ]:
# now we have to make all he 36 length so it will be helpful for the batch learning 

padded_training_sequence=[]
for sequence in training_sequence:
    # print([0]*(max(len_list)-len(sequence)) + sequence)
    padded_training_sequence.append([0]*(max(len_list)-len(sequence)) + sequence)


In [49]:
padded_training_sequence[0]
len(padded_training_sequence[11])

36

In [ ]:
# converting to tensors

padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

padded_training_sequence.shape # converted to tensors

/tmp/ipykernel_36493/4285233185.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)


torch.Size([974, 36])

In [55]:
X = padded_training_sequence[:,:-1] # all rows except last 

y = padded_training_sequence[:, -1]
y

tensor([  2,   3,   4,   5,   6,   7,   9,  10,   6,  11,   5,   2,   3,   4,
          7,   9,  12,  13,  14,  15,  16,  17,  18,   5,  19,   7,  20,  21,
         22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  15,
         35,  36,  28,  38,   6,  39,  11,  36,  40,  21,  41,  42,  43,   2,
         44,  45,  21,  32,  46,  28,  47,  48,   4,  49,  50,  51,  52,  21,
         53,  54,  42,  55,  56,  57,  28,  59,   6,  11,  38,  39,  60,  61,
         62,  50,  63,  64,  25,   0,  28,  66,  67,  47,  68,  11,  69,  70,
         71,  55,  72,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,
         81,  85,  86,  87,  88,  89,   9,  31,  90,  91,  92,  19,  50,   6,
         11,  19,  93,  94,  12,  95,  96,  97,  98,  10,   6,  94,  99,  12,
         95,  96,  50,  12, 100,  96,  97, 101,  29,  21,  41,  42,  43,   2,
         44,  28,  69,  31,  90, 102,  15,  10,   6, 100,  11,   6, 103,  15,
         72, 104, 105, 106, 105, 107, 100,  96, 108, 105,  95,  

In [56]:
class customDataset(Dataset):

    def __init__(self,X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [59]:
dataset = customDataset(X,y)
len(dataset)

974

In [66]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [64]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
for input,output in dataloader:
    print(input,output)

tensor([[  0,   0,   0,  ..., 166,  15, 167],
        [  0,   0,   0,  ...,   0,   0,   8],
        [  0,   0,   0,  ...,   0, 104,  31],
        ...,
        [  0,   0,   0,  ..., 120, 126,  31],
        [  0,   0,   0,  ...,   0,   0,   8],
        [  0,   0,   0,  ..., 108, 105, 146]]) tensor([  9, 131,  42, 234,  11, 203,  67,  97,  97, 193,   7,  81,  77, 279,
          9, 130,  11, 100,  28,  47,  11, 138,  11,  11,  95,  99, 188,  14,
         49, 237,   9,   8])
tensor([[  0,   0,   0,  ...,   0, 128, 129],
        [  0,   0,   0,  ...,  95, 174,  69],
        [  0,   0,   0,  ...,   0,  73, 258],
        ...,
        [  0,   0,   0,  ...,   0,  12, 165],
        [  0,   0,   0,  ...,  94,  12,  95],
        [  0,   0,   0,  ...,   0, 144, 145]]) tensor([130,  70, 259, 122,  69, 131, 147, 141,  97, 252,  34, 172, 221, 191,
         94,  11, 105,  31,  82,  28,  31, 147, 174,  21,  16,  28,  28,  93,
          9, 166,  96,  31])
tensor([[  0,   0,   0,  ..., 217,  15, 223],
    

In [67]:
X.shape

torch.Size([974, 35])

In [68]:
y.shape

torch.Size([974])

In [118]:
# X has (974,35) and y has 974 
# and we are using the batches of 32 ie 

# 32 sentences in each batch. and each sentence is represented in 100 dimensions

class LSTMModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,100)
        self.lstm = nn.LSTM(100,150, batch_first=True)
        self.fc = nn.Linear(150, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
        output = self.fc(final_hidden_state.squeeze(0))

        return output        

## what happens in each layers 

In [81]:
    w = nn.Embedding(313,100)
    v = nn.LSTM(100,150, batch_first=True)
    z=nn.Linear(150, 313)

In [71]:
a = dataset[0][0].unsqueeze(0)

In [ ]:
b=w(a)
c,d=v(b)
# c is set of all intermediate hidden state

In [77]:
d

(tensor([[[ 3.3815e-02, -8.0228e-02,  9.1508e-02,  3.9899e-03, -1.9441e-01,
            7.6950e-02, -1.3776e-01, -1.3543e-01, -1.5011e-01,  2.5630e-02,
           -2.3889e-01, -1.5488e-01,  1.3957e-01,  1.1479e-01, -8.9516e-02,
           -2.4086e-01, -1.8545e-01, -8.6354e-02,  2.0234e-01,  1.7624e-01,
           -1.3773e-01,  6.0064e-02, -1.3181e-01, -1.1882e-01, -1.6908e-01,
            3.0299e-03, -9.4555e-03, -2.3272e-02,  5.7015e-02, -9.1338e-02,
           -1.4712e-01,  4.4200e-02,  1.7437e-01, -6.0178e-02,  2.2089e-02,
           -9.2936e-03, -1.0927e-01, -7.9732e-02,  1.1114e-01, -2.3649e-01,
           -1.0365e-01,  1.5875e-01, -3.7866e-03, -1.7929e-02, -1.2118e-01,
           -1.1275e-01, -6.8653e-02,  1.8711e-02,  1.1224e-02,  8.8814e-02,
            2.5402e-01, -2.8848e-01, -1.3515e-01, -1.0503e-01, -1.8102e-01,
            1.8962e-01, -3.9243e-01, -9.3216e-02, -5.0503e-02, -4.5415e-02,
           -5.1060e-02, -6.1493e-02,  3.9572e-02, -3.1263e-01, -5.4406e-02,
           -

In [76]:
c.shape

torch.Size([1, 35, 150])

So we sent 1 sequence sequence , 35 hidden layers, 150 dimensions

In [78]:
e,f = d
e.shape

torch.Size([1, 1, 150])

In [ ]:
z(f.squeeze(0)).shape
# this if final output layer which give proba for each 313 vocab layer 

torch.Size([1, 313])

# Model formation 

In [119]:
model = LSTMModel(len(vocab))


In [120]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [121]:
model.to(device)

LSTMModel(
  (embedding): Embedding(313, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=313, bias=True)
)

In [122]:
epochs = 100

learning_rate = 0.001

In [123]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [124]:
# training loop 

for epoch in range(epochs):

    total_loss = 0

    for batch_x , batch_y in dataloader:

        batch_x,batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        output = model(batch_x)

        loss = criterion(output, batch_y)

        loss.backward()

        optimizer.step()

        total_loss = total_loss + loss.item()

    print(f"Epoch : {epoch + 1 } , Loss : {total_loss:.4f}")



Epoch : 1 , Loss : 171.9474
Epoch : 2 , Loss : 149.6632
Epoch : 3 , Loss : 136.7430
Epoch : 4 , Loss : 124.4581
Epoch : 5 , Loss : 112.9449
Epoch : 6 , Loss : 102.0470
Epoch : 7 , Loss : 92.3345
Epoch : 8 , Loss : 83.5848
Epoch : 9 , Loss : 75.5590
Epoch : 10 , Loss : 67.8710
Epoch : 11 , Loss : 60.9830
Epoch : 12 , Loss : 54.6179
Epoch : 13 , Loss : 49.3091
Epoch : 14 , Loss : 43.8517
Epoch : 15 , Loss : 39.4152
Epoch : 16 , Loss : 35.7778
Epoch : 17 , Loss : 32.9744
Epoch : 18 , Loss : 30.0905
Epoch : 19 , Loss : 27.3191
Epoch : 20 , Loss : 25.3620
Epoch : 21 , Loss : 23.8889
Epoch : 22 , Loss : 22.3814
Epoch : 23 , Loss : 21.0940
Epoch : 24 , Loss : 20.1918
Epoch : 25 , Loss : 19.3737
Epoch : 26 , Loss : 18.5880
Epoch : 27 , Loss : 17.7822
Epoch : 28 , Loss : 17.3179
Epoch : 29 , Loss : 16.9806
Epoch : 30 , Loss : 16.5587
Epoch : 31 , Loss : 16.0670
Epoch : 32 , Loss : 15.6102
Epoch : 33 , Loss : 15.5686
Epoch : 34 , Loss : 15.2257
Epoch : 35 , Loss : 14.9183
Epoch : 36 , Loss : 14.

In [149]:
# prediction 

def prediction(model, vocab, text ):
    # tokenize 
    tokenized_text = word_tokenize(text.lower())
    # print(tokenized_text)

    # text -> numerical indices
    numerical_text = text_to_indices(tokenized_text,vocab)
    # print(numerical_text)

    # padding 
    padded_text = torch.tensor([0]*(35 - len(numerical_text))+ numerical_text, dtype=torch.long).unsqueeze(0)
    # print(padded_text.shape)

    padded_text= padded_text.to(device)
    # send to model
    output = model(padded_text)
    # print(output)
    # print(output.shape)



    # predicted index
    value , index = torch.max(output, dim=1)
    # print(index)
    # merge with the text 

    return text + " " + list(vocab.keys())[index]

In [150]:
prediction(model, vocab, "The Forget Gate decides")

'The Forget Gate decides what'

In [152]:
import time
num_tokens = 10 

input_text = "Why does LSTM perform better"


for i in range(num_tokens):
    output_text = prediction(model,vocab, input_text)
    print(output_text)
    input_text=output_text
    time.sleep(0.5)

Why does LSTM perform better than
Why does LSTM perform better than rnn
Why does LSTM perform better than rnn ?
Why does LSTM perform better than rnn ? ?
Why does LSTM perform better than rnn ? ? and
Why does LSTM perform better than rnn ? ? and lstm
Why does LSTM perform better than rnn ? ? and lstm .
Why does LSTM perform better than rnn ? ? and lstm . it
Why does LSTM perform better than rnn ? ? and lstm . it carries
Why does LSTM perform better than rnn ? ? and lstm . it carries important


The error occurs because **your model is on the GPU (`cuda:0`) but your input tensor (`padded_text`) is on the CPU**.

PyTorch requires that **both the model and all input tensors be on the same device**.

---

## Why this happens

Your code creates the tensor like this:

```python
padded_text = torch.tensor(
    [0] * (61 - len(numerical_text)) + numerical_text,
    dtype=torch.long
).unsqueeze(0)
```

By default, `torch.tensor()` creates a **CPU tensor**.

But your model is probably on the GPU:

```python
model.to(device)   # device = cuda
```

So when you do

```python
output = model(padded_text)
```

PyTorch complains because

```
Model : CUDA
Input : CPU
```

---

# Fix 1 (Recommended)

Move the input tensor to the same device as the model.

```python
device = next(model.parameters()).device
```

Then

```python
padded_text = torch.tensor(
    [0] * (61 - len(numerical_text)) + numerical_text,
    dtype=torch.long
).unsqueeze(0).to(device)
```

This automatically works whether the model is on CPU or GPU.

---

# Fix 2

If you already have

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

then

```python
padded_text = padded_text.to(device)
```

---

# Another Bug in Your Code

This line is also problematic:

```python
list(vocab.keys())[index]
```

`index` is **a tensor**, not an integer.

After

```python
value, index = torch.max(output, dim=1)
```

you get

```
index = tensor([5], device='cuda:0')
```

You need

```python
predicted_class = index.item()
```

Then

```python
list(vocab.keys())[predicted_class]
```

---

# Complete Correct Function

```python
def prediction(model, vocab, text):

    device = next(model.parameters()).device

    # Tokenize
    tokenized_text = word_tokenize(text.lower())

    # Convert to indices
    numerical_text = text_to_indices(tokenized_text, vocab)

    # Padding
    padded_text = torch.tensor(
        [0] * (61 - len(numerical_text)) + numerical_text,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    # Inference mode
    model.eval()
    with torch.no_grad():
        output = model(padded_text)

    # Predicted index
    _, index = torch.max(output, dim=1)

    predicted_idx = index.item()

    return text + " " + list(vocab.keys())[predicted_idx]
```

---

# If This Is a Language Model

If your model predicts the **next word**, then using

```python
list(vocab.keys())[predicted_idx]
```

is **not recommended** because Python dictionaries are not guaranteed to preserve the exact vocabulary ordering used during training (unless you explicitly built them that way).

Instead, create an inverse vocabulary once:

```python
idx_to_word = {idx: word for word, idx in vocab.items()}
```

Then retrieve the prediction with:

```python
predicted_word = idx_to_word[predicted_idx]
```

This guarantees that the predicted index maps back to the correct word.

If you share your **LSTM model class** and how you created `vocab`, I can also verify that the output layer size and index-to-word mapping are correct.
